Second experiment:

Ornstein-Uhlenback

In [3]:
#!pip install matplotlib

In [2]:
"""Experiments I and II: geometric Brownian motion and OU moments.
Run: python split_nsde_experiments_modified.py
Dependencies: numpy, matplotlib. Reproducible fixed seeds.
"""
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

# Current working directory (Jupyter Notebook or VS Code)
OUT = Path.cwd() / "split_nsde_results"

# Create the directory if it does not exist
OUT.mkdir(parents=True, exist_ok=True)

print("Results will be saved in:", OUT)


#OUT = Path(__file__).resolve().parent / 'split_nsde_results'
#OUT.mkdir(exist_ok=True)
GRIDS = [8, 16, 32, 64, 128, 256]


def gbm(nsteps, paths=20000, mu=-0.4, sigma=0.7, T=1., seed=20261010):
    """Pathwise-coupled GBM with exact solution; no artificial error floor."""
    rng = np.random.default_rng(seed)
    h = T/nsteps
    em = np.ones(paths); lie = np.ones(paths); strang = np.ones(paths)
    W = np.zeros(paths)
    for _ in range(nsteps):
        dw = rng.normal(0, np.sqrt(h), paths)
        W += dw
        em *= 1 + mu*h + sigma*dw
        lie *= np.exp((mu-0.5*sigma**2)*h + sigma*dw)
        strang *= np.exp(mu*h/2)*np.exp(-0.5*sigma**2*h+sigma*dw)*np.exp(mu*h/2)
    exact = np.exp((mu-0.5*sigma**2)*T + sigma*W)
    return {name: {'strong_rmse': float(np.sqrt(np.mean((z-exact)**2))),
                   'sample_mean': float(z.mean()),
                   'weak_mean_bias': float(abs(z.mean()-exact.mean()))}
            for name,z in [('Euler-Maruyama',em),('Lie',lie),('Strang',strang)]}


def ou_exact_moments(lam=3., m=1., sigma=0.8, z0=0., T=2.):
    mean = m + (z0-m)*np.exp(-lam*T)
    variance = sigma**2/(2*lam)*(-np.expm1(-2*lam*T))
    return {'mean': float(mean), 'variance': float(variance),
            'second_moment': float(variance+mean**2)}


def ou_theoretical_split_moments(nsteps, method, lam=3., m=1., sigma=0.8, z0=0., T=2.):
    """Exact moments of each numerical scheme (not Monte Carlo estimates)."""
    h = T/nsteps; q = np.exp(-lam*h)
    if method == 'Euler-Maruyama':
        a = 1-lam*h; b = lam*h*m; noise_var = sigma**2*h
    elif method == 'Lie':
        a = q; b = m*(1-q); noise_var = sigma**2*h
    elif method == 'Strang':
        a = q; b = m*(1-q); noise_var = sigma**2*h*q
    else:
        raise ValueError(method)
    mean = z0; var = 0.
    for _ in range(nsteps):
        mean = a*mean+b
        var = a*a*var+noise_var
    return {'mean': mean, 'variance': var, 'second_moment': var+mean**2}


def ou(nsteps, paths=100000, lam=3., m=1., sigma=0.8, z0=0., T=2., seed=20261010):
    """Common Brownian increments for all three methods; exact OU sampled with
    correlated increments for pathwise error comparisons. OU noise integral
    conditional on dW has covariance sigma*(1-exp(-lam*h))/lam.
    """
    rng = np.random.default_rng(seed)
    h = T/nsteps; q = np.exp(-lam*h); qhalf = np.exp(-lam*h/2)
    exact = np.full(paths,z0,dtype=float)
    states = {key: np.full(paths,z0,dtype=float) for key in
              ('Euler-Maruyama','Lie','Strang')}
    integral_var = (-np.expm1(-2*lam*h))/(2*lam)
    cov = (-np.expm1(-lam*h))/lam
    coefficient = cov/h
    residual_std = np.sqrt(max(0., integral_var-cov**2/h))
    for _ in range(nsteps):
        dw = rng.normal(0,np.sqrt(h),paths)
        independent = rng.normal(0,1,paths)
        stochastic_integral = coefficient*dw+residual_std*independent
        exact = m+(exact-m)*q+sigma*stochastic_integral
        states['Euler-Maruyama'] += -lam*(states['Euler-Maruyama']-m)*h+sigma*dw
        states['Lie'] = m+(states['Lie']-m)*q+sigma*dw
        states['Strang'] = m+(states['Strang']-m)*q+sigma*qhalf*dw
    truth = ou_exact_moments(lam,m,sigma,z0,T)
    results = {}
    for name,z in states.items():
        theory = ou_theoretical_split_moments(nsteps,name,lam,m,sigma,z0,T)
        sample = {'mean':float(z.mean()),'variance':float(z.var()),
                  'second_moment':float(np.mean(z*z))}
        results[name] = {
            **{f'sample_{k}':v for k,v in sample.items()},
            **{f'theory_{k}':float(v) for k,v in theory.items()},
            **{f'sample_{k}_bias':abs(sample[k]-truth[k]) for k in truth},
            **{f'theory_{k}_bias':abs(theory[k]-truth[k]) for k in truth},
            'strong_rmse':float(np.sqrt(np.mean((z-exact)**2))),
            'w2_empirical_1d':float(np.sqrt(np.mean((np.sort(z)-np.sort(exact))**2))),
        }
    return results


def save_csv(filename,rows):
    with (OUT/filename).open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)


def plot_metric(rows, metric, title, filename, ylabel):
    fig,ax=plt.subplots(figsize=(7,4.5))
    for method in ('Euler-Maruyama','Lie','Strang'):
        subset=[r for r in rows if r['method']==method]
        ax.loglog([r['h'] for r in subset],
                  [max(r[metric],1e-16) for r in subset], 'o-',label=method)
    ax.set(xlabel='Step size h',ylabel=ylabel,title=title)
    ax.grid(True,which='both',alpha=.3);ax.legend();fig.tight_layout()
    fig.savefig(OUT/filename,dpi=180);plt.close(fig)


def main():
    gbm_rows=[];ou_rows=[]
    print('OU exact moments:',ou_exact_moments())
    for n in GRIDS:
        print(f'Computing n={n} ...')
        for method,r in gbm(n).items():
            gbm_rows.append({'nsteps':n,'h':1/n,'method':method,**r})
        for method,r in ou(n).items():
            ou_rows.append({'nsteps':n,'h':2/n,'method':method,**r})
    save_csv('gbm_results.csv',gbm_rows)
    save_csv('ou_moments.csv',ou_rows)
    plot_metric(gbm_rows,'strong_rmse','GBM strong convergence','gbm_strong.png','Strong RMSE')
    for moment in ('mean','variance','second_moment'):
        plot_metric(ou_rows,f'theory_{moment}_bias',f'OU {moment}: numerical weak bias',
                    f'ou_{moment}_bias.png',f'Absolute {moment} bias (analytic)')
    plot_metric(ou_rows,'strong_rmse','OU coupled strong convergence','ou_strong.png','Strong RMSE')
    print('\nOU theoretical moment biases (no Monte Carlo noise):')
    for r in ou_rows:
        if r['nsteps'] in (8,32,128,256):
            print(f"n={r['nsteps']:3d} {r['method']:16s} mean={r['theory_mean_bias']:.3e} "
                  f"variance={r['theory_variance_bias']:.3e} second={r['theory_second_moment_bias']:.3e}")
    print('\nSaved tables and figures in',OUT)

if __name__=='__main__':
    main()


Results will be saved in: c:\Users\geise\OneDrive\Desktop\Research_2025\Paper_ICICT_2026\Code_Experiments\split_nsde_results
OU exact moments: {'mean': 0.9975212478233336, 'variance': 0.10666601128401566, 'second_moment': 1.1017146511430362}
Computing n=8 ...
Computing n=16 ...
Computing n=32 ...
Computing n=64 ...
Computing n=128 ...
Computing n=256 ...

OU theoretical moment biases (no Monte Carlo noise):
n=  8 Euler-Maruyama   mean=2.463e-03 variance=6.400e-02 second=6.892e-02
n=  8 Lie              mean=1.110e-16 variance=9.929e-02 second=9.929e-02
n=  8 Strang           mean=1.110e-16 variance=9.380e-03 second=9.380e-03
n= 32 Euler-Maruyama   mean=1.178e-03 variance=1.103e-02 second=1.339e-02
n= 32 Lie              mean=0.000e+00 variance=2.125e-02 second=2.125e-02
n= 32 Strang           mean=0.000e+00 variance=6.224e-04 second=6.224e-04
n=128 Euler-Maruyama   mean=3.350e-04 variance=2.560e-03 second=3.229e-03
n=128 Lie              mean=1.110e-16 variance=5.078e-03 second=5.078e-